# Tick-built vs minute-approximated bars

Quantifies how much bar construction diverges when the input is downsampled
from raw ticks to minute OHLCV candles. Both sides use flowbars — the tick path
(`compute_*_bars`) vs the minute path (`compute_*_bars_from_minutes`) — on the
same underlying data.

The minute path is an approximation by design (see SPEC, "Minute-OHLCV → bars
path"). Four sources of divergence are measured here:

1. **Boundary granularity** — minute bars can only close at minute boundaries
   (whole-minute attribution); tick bars close at the exact crossing tick.
2. **VWAP proxy** — the minute path uses `close × volume` as notional, not the
   exact `Σ price × volume`.
3. **`num_ticks` = minutes** — trade count is lost; the field counts minutes.
4. **Minute tick-rule signs** — one sign per minute (close-to-close) nets out
   intra-minute reversals, reducing directional information.


## Data

Synthetic ticks: 300 minutes, 10–50 ticks/minute, with intra-minute price moves
so the minute high/low genuinely differ from the close. The same ticks are then
aggregated into minute OHLCV candles.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from flowbars import (
    SchemaMapping,
    compute_dollar_bars,
    compute_imbalance_volume_bars,
    compute_run_volume_bars,
    compute_volume_bars,
)
from flowbars.bars.minute import (
    compute_dollar_bars_from_minutes,
    compute_imbalance_volume_bars_from_minutes,
    compute_run_volume_bars_from_minutes,
    compute_volume_bars_from_minutes,
)
from flowbars.schema import MinuteSchemaMapping
from flowbars.tick_rule import derive_tick_sign

rng = np.random.default_rng(7)

rows = []
for m in range(300):
    n = int(rng.integers(10, 50))
    t0 = m * 60_000
    offsets = np.sort(rng.integers(0, 60_000, n))
    base = 100.0 + 0.02 * m + rng.normal(0, 0.1)
    prices = base + np.cumsum(rng.normal(0, 0.05, n))
    volumes = rng.integers(1, 10, n).astype(float)
    for i in range(n):
        rows.append((int(t0 + offsets[i]), float(prices[i]), float(volumes[i])))

ticks = pd.DataFrame(rows, columns=["ts", "price", "volume"])

minute = (
    ticks.assign(minute=ticks["ts"] // 60_000)
    .groupby("minute")
    .agg(
        ts=("ts", "min"),
        open=("price", "first"),
        high=("price", "max"),
        low=("price", "min"),
        close=("price", "last"),
        volume=("volume", "sum"),
    )
    .reset_index(drop=True)
)

TICK_SCHEMA = SchemaMapping({"timestamp": "ts", "price": "price", "volume": "volume"})
MIN_SCHEMA = MinuteSchemaMapping(
    {"timestamp": "ts", "open": "open", "high": "high", "low": "low", "close": "close", "volume": "volume"}
)

TOTAL_VOL = float(ticks["volume"].sum())
TOTAL_DOL = float((ticks["price"] * ticks["volume"]).sum())

print(f"ticks: {len(ticks):,} | minutes: {len(minute):,}")
print(f"total volume: {TOTAL_VOL:,.0f} | total dollar (exact): {TOTAL_DOL:,.0f}")


## Finding 1 — standard bars: bar count + VWAP proxy error

Same threshold on both paths. `vwap_err_%` isolates the `close × volume` proxy:
it is the relative error introduced by pricing every trade in a minute at the
minute's close instead of its actual price.


In [ ]:
V_THRESH = TOTAL_VOL / 20.0
D_THRESH = TOTAL_DOL / 20.0


def _vwap(df):
    v = float(df["volume"].sum())
    return float(df["dollar_value"].sum()) / v if v else float("nan")


rows = []
for name, tick_fn, min_fn, thresh in [
    ("volume", compute_volume_bars, compute_volume_bars_from_minutes, V_THRESH),
    ("dollar", compute_dollar_bars, compute_dollar_bars_from_minutes, D_THRESH),
]:
    tb = tick_fn(ticks, threshold=thresh, schema=TICK_SCHEMA, watermark=None)
    mb = min_fn(minute, threshold=thresh, schema=MIN_SCHEMA)
    vw_t, vw_m = _vwap(tb), _vwap(mb)
    rows.append(
        {
            "bar_type": name,
            "tick_bars": len(tb),
            "minute_bars": len(mb),
            "vwap_tick": vw_t,
            "vwap_minute": vw_m,
            "vwap_err_%": 100.0 * (vw_m - vw_t) / vw_t,
        }
    )

pd.DataFrame(rows)


## Finding 2 — signs are a different signal, not a lossy one

Tick signs come from the tick rule on every trade; minute signs come from the
tick rule on minute closes. These measure different things:

- **Tick-level** signs alternate rapidly (up/down within each minute), so the
  total signed volume nets toward zero and runs are short.
- **Minute-level** signs capture the net close-to-close direction, which follows
  the slower price drift — so the signed volume can differ in magnitude *and
  even sign*.

The result: imbalance/run bars built from minutes are **not** a coarser version
of the tick bars — they carry a genuinely different directional signal.


In [ ]:
tick_signs = np.nan_to_num(derive_tick_sign(ticks["price"].to_numpy()), nan=0.0)
min_signs = np.nan_to_num(derive_tick_sign(minute["close"].to_numpy()), nan=0.0)

tick_signed_vol = float((tick_signs * ticks["volume"]).sum())
min_signed_vol = float((min_signs * minute["volume"]).sum())

print(f"total signed volume — ticks: {tick_signed_vol:,.1f} | minutes: {min_signed_vol:,.1f}")
print("note: these can differ in magnitude and sign — different signals, not a lossy copy")


info = [
    ("imbalance_volume", compute_imbalance_volume_bars, compute_imbalance_volume_bars_from_minutes),
    ("run_volume", compute_run_volume_bars, compute_run_volume_bars_from_minutes),
]

rows = []
for name, tick_fn, min_fn in info:
    tb = tick_fn(ticks, span=20.0, schema=TICK_SCHEMA, watermark=None)
    mb = min_fn(minute, span=20.0, schema=MIN_SCHEMA)
    rows.append({"bar_type": name, "tick_bars": len(tb), "minute_bars": len(mb)})

pd.DataFrame(rows)


## Interpretation

- **Bar count** diverges because minute bars close at the end of a minute, never
  mid-minute — the coarser the granularity relative to bar size, the larger the
  count gap (dollar: 20 vs 19).
- **VWAP error** is the `close × volume` proxy. It is small (< 0.15%) because
  within-minute prices stay close to the minute close, but it is not zero.
- **Directional signal** is the biggest divergence: the minute tick-rule
  (close-to-close) and the tick tick-rule (trade-to-trade) are different
  signals. Tick signs net near zero and give short runs; minute signs follow
  the drift and give long runs. Downsampling changes the *information content*
  of imbalance/run bars, not just their resolution.
